In [ ]:
import pandas as pd

df_parent = pd.read_parquet("question_answer/parent.parquet")
df_self = pd.read_parquet("question_answer/self.parquet")

In [ ]:
for elem in df_parent.open_text_block.to_list():
    print(f"{elem}\n\n******\n\n")

In [ ]:
for elem in df_self.open_text_block.to_list():
    print(f"{elem}\n\n******\n\n")

In [ ]:
import re


def format_qa_block(text, name_replacement, role):
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.replace("\r\n", "\n").replace("__FIRSTNAME__", name_replacement)
    pairs = re.findall(
        r"Question\s*:\s*(.*?)\s*Answer\s*:\s*(.*?)(?=\n\s*Question\s*:|\Z)",
        text,
        re.DOTALL | re.IGNORECASE,
    )
    blocks = []
    for i, (q, a) in enumerate(pairs, start=1):
        blocks.append(f"Q{i}: {q.strip()}\nA{i} ({role}): {a.strip()}")
    return "\n\n".join(blocks)


df_parent["formatted_text"] = df_parent["open_text_block"].apply(
    lambda t: format_qa_block(t, "My Child", "Parent")
)

df_self["formatted_text"] = df_self["open_text_block"].apply(
    lambda t: format_qa_block(t, "[my_name]", "Self")
)

In [ ]:
for elem in df_self.formatted_text.to_list():
    print(f"{elem}\n\n******\n\n")

In [ ]:
for elem in df_parent.formatted_text.to_list():
    print(f"{elem}\n\n******\n\n")

In [ ]:
df_parent.to_csv("question_answer/df_parent_formatted.csv", index=False)
df_self.to_csv("question_answer/df_self_formatted.csv", index=False)

In [ ]:
df_parent = pd.read_csv("question_answer/df_parent_formatted.csv")
df_self = pd.read_csv("question_answer/df_self_formatted.csv")